# 34 — Data Services: RDS PostgreSQL & ClickHouse

**Role:** Senior AWS DevOps Engineer | **Focus:** RDS Management, PostgreSQL Ops, ClickHouse, Database DevOps

---

## Section A: RDS PostgreSQL Operations

### Q1: RDS Architecture Decisions
**Question:** You need to deploy a PostgreSQL database for a production SaaS platform. Walk through your decisions: Multi-AZ, Read Replicas, instance sizing, and storage.

**Expected Answer:**
- **Multi-AZ**: Always for production. Synchronous standby in another AZ. Automatic failover (60–120s).
- **Read Replicas**: For read-heavy workloads. Async replication. Can be cross-region for DR.
- **Instance**: Start with `db.r6g.large` (Graviton, memory-optimized). Use Performance Insights to right-size.
- **Storage**: `gp3` (baseline 3000 IOPS, scalable). Avoid `gp2` for predictable performance. `io2` for extreme IOPS needs.
- **Encryption**: Enable at creation with CMK. Cannot add later.
- **Parameter Group**: Tune `shared_buffers`, `work_mem`, `max_connections` based on workload.

---

### Q2: Backup & Recovery
**Question:** Your production RDS instance is corrupted due to a bad migration. How do you recover?

**Expected Answer:**
- **Automated backups**: RDS takes daily snapshots + stores WAL logs. Retention up to 35 days.
- **Point-in-Time Recovery (PITR)**: Restore to any second within the retention window. Creates a NEW instance.
- **Process**: Restore PITR → verify data → promote to primary → update DNS/connection strings.
- **Manual snapshots**: Take before every migration. No retention limit.
- **Cross-region**: Copy snapshots to another region for DR.
- **RTO/RPO**: PITR gives RPO of ~5 minutes (WAL upload interval). RTO depends on DB size.

---

### Q3: Performance Troubleshooting
**Question:** Application latency spikes and you suspect the database. How do you diagnose?

**Expected Answer:**
- **Performance Insights**: Top SQL by wait events, load by waits.
- **Common waits**: `IO:DataFileRead` (need more IOPS or memory), `Lock:transactionid` (row-level contention), `CPU` (inefficient queries).
- **`pg_stat_statements`**: Find slow queries by mean/total execution time.
- **Connection pooling**: If seeing `Client:ClientRead` waits, use PgBouncer or RDS Proxy.
- **Vacuum issues**: Bloated tables from missing autovacuum. Check `pg_stat_user_tables` for dead tuples.
- **Index analysis**: `EXPLAIN ANALYZE` on slow queries. Missing indexes = sequential scans on large tables.

## Section B: Database DevOps

### Q4: Schema Migrations in CI/CD
**Question:** How do you safely run schema migrations on a production PostgreSQL database via GitLab CI?

**Expected Answer:**
- **Tool**: Alembic (Python), Flyway (Java/general), or Sqitch.
- **Expand-Contract pattern**: (1) Add new column (nullable). (2) Deploy app that writes to both. (3) Backfill. (4) Switch reads. (5) Drop old column.
- **Non-locking DDL**: `CREATE INDEX CONCURRENTLY`, `ALTER TABLE ... ADD COLUMN` (nullable is instant in PG 11+).
- **Pipeline integration**: Migration runs as a dedicated CI stage BEFORE app deployment.
- **Rollback**: Every migration has a `down` script. Tested in staging.
- **Connection**: CI runner connects via VPC peering or VPN. Credentials from Secrets Manager.

---

### Q5: RDS Proxy & Connection Management
**Question:** Why would you use RDS Proxy in front of PostgreSQL? What problem does it solve?

**Expected Answer:**
- **Problem**: PostgreSQL has a process-per-connection model. Too many connections = OOM or CPU saturation.
- **Lambda**: Each Lambda invocation opens a new connection. 1000 concurrent Lambdas = 1000 DB connections.
- **RDS Proxy**: Connection pooling + multiplexing. Maintains a warm pool, reuses connections.
- **Failover**: RDS Proxy also provides faster failover (~20s vs ~60s) during Multi-AZ switch.
- **IAM Auth**: RDS Proxy supports IAM authentication — no passwords in code.
- **Alternative**: Self-managed PgBouncer on EC2 or as a K8s sidecar (more control, more ops).

## Section C: ClickHouse

### Q6: ClickHouse vs. PostgreSQL
**Question:** When would you choose ClickHouse over PostgreSQL? What are the key architectural differences?

**Expected Answer:**
- **ClickHouse**: Column-oriented OLAP database. Designed for analytical queries over billions of rows.
- **PostgreSQL**: Row-oriented OLTP. Best for transactional workloads (CRUD, joins, constraints).
- **Use ClickHouse for**: Event analytics, log storage, time-series data, real-time dashboards.
- **Performance**: ClickHouse can scan 1B rows in seconds. PostgreSQL would struggle.
- **Trade-offs**: ClickHouse has limited UPDATE/DELETE (not ACID), no foreign keys, eventual consistency in distributed mode.
- **Ingestion**: Batch inserts preferred (not row-by-row). Kafka → ClickHouse is common.

---

### Q7: ClickHouse Operations
**Question:** How do you deploy and scale ClickHouse on AWS? What are the operational challenges?

**Expected Answer:**
- **Deployment**: EC2 instances with local NVMe (i3/i4i) for performance, or EBS gp3.
- **Clustering**: ReplicatedMergeTree engine + ZooKeeper/ClickHouse Keeper for coordination.
- **Sharding**: Distributed table across shards. Shard key choice is critical (avoid hotspots).
- **Scaling**: Add shards for write throughput, replicas for read throughput.
- **Challenges**: No managed AWS service (unlike RDS). Must manage upgrades, backups, monitoring yourself.
- **Alternatives**: Amazon Timestream (time-series), Athena (S3 queries), or ClickHouse Cloud (managed).